In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA is unavailable; check the runtime type."
print("GPU:", torch.cuda.get_device_name(0))
print("Free/total GPU memory:", torch.cuda.mem_get_info())

GPU: NVIDIA L4
Free/total GPU memory: (23455727616, 23659151360)


In [ ]:
%pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199

In [ ]:
import importlib.util

if importlib.util.find_spec("unsloth") is None:
    raise RuntimeError(
        "Unsloth is not installed. In a separate cell run `%pip install -q unsloth`, "
        "restart the runtime, then start again from Cell 1."
    )

print("Unsloth is installed.")

Unsloth is installed.


In [ ]:
import gc
import torch
from pathlib import Path
from zipfile import ZipFile

from datasets import load_dataset
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

assert torch.cuda.is_available(), "CUDA is unavailable. Select L4 and reconnect."

gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU: {gpu_name} | VRAM: {gpu_gib:.1f} GiB")

assert "L4" in gpu_name.upper(), (
    f"Expected an L4, but Colab assigned {gpu_name}. Stop here."
)
assert gpu_gib >= 20, f"Expected about 24 GB GPU memory, found {gpu_gib:.1f} GiB."

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1606: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: NVIDIA L4 | VRAM: 22.0 GiB


In [ ]:
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/FinRAG_Training")
PROJECT_DIR = Path("/content/finrag")
DATA_DIR = PROJECT_DIR / "fine_tuning" / "work" / "sft_final_800"
ZIP_PATH = DRIVE_ROOT / "finrag_lora_ready_800_20261002.zip"

splits = ("train", "validation", "test")

if not all((DATA_DIR / f"{split}.jsonl").is_file() for split in splits):
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f"Dataset archive not found: {ZIP_PATH}")

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with ZipFile(ZIP_PATH) as archive:
        archive.extractall(PROJECT_DIR)

missing = [
    split for split in splits
    if not (DATA_DIR / f"{split}.jsonl").is_file()
]
if missing:
    raise FileNotFoundError(f"Missing dataset split files: {missing}")

data = load_dataset(
    "json",
    data_files={
        split: str(DATA_DIR / f"{split}.jsonl")
        for split in splits
    },
)

expected_counts = {"train": 692, "validation": 68, "test": 40}
for split, expected in expected_counts.items():
    actual = len(data[split])
    print(f"{split}: {actual}")
    assert actual == expected, f"Expected {expected} {split} examples, found {actual}."

Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

train: 692
validation: 68
test: 40


In [ ]:
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/FinRAG_Training")
PROJECT_DIR = Path("/content/finrag")
DATA_DIR = PROJECT_DIR / "fine_tuning" / "work" / "sft_final_800"
ZIP_PATH = DRIVE_ROOT / "finrag_lora_ready_800_20261002.zip"

splits = ("train", "validation", "test")

if not all((DATA_DIR / f"{split}.jsonl").is_file() for split in splits):
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f"Dataset archive not found: {ZIP_PATH}")

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with ZipFile(ZIP_PATH) as archive:
        archive.extractall(PROJECT_DIR)

missing = [
    split for split in splits
    if not (DATA_DIR / f"{split}.jsonl").is_file()
]
if missing:
    raise FileNotFoundError(f"Missing dataset split files: {missing}")

data = load_dataset(
    "json",
    data_files={
        split: str(DATA_DIR / f"{split}.jsonl")
        for split in splits
    },
)

expected_counts = {"train": 692, "validation": 68, "test": 40}
for split, expected in expected_counts.items():
    actual = len(data[split])
    print(f"{split}: {actual}")
    assert actual == expected, f"Expected {expected} {split} examples, found {actual}."

train: 692
validation: 68
test: 40


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 5376

def load_model_and_lora():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        device_map={"": 0},
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    model.config.use_cache = False

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        max_seq_length=MAX_SEQ_LENGTH,
    )
    return model, tokenizer

model, tokenizer = load_model_and_lora()
print("Base model and LoRA loaded.")

==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.14 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers. The fused LoRA kernels were skipped because lora_dropout = 0.05, which is why the counts are zero. Training is unaffected.


Base model and LoRA loaded.


In [ ]:
lengths = {}

for split in splits:
    lengths[split] = []
    for row in data[split]:
        messages = row["prompt"] + row["completion"]
        token_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
        )
        lengths[split].append(len(token_ids))

    print(f"{split}: {len(lengths[split])} examples; max={max(lengths[split])} tokens")

longest = max(max(lengths[split]) for split in splits)
assert longest <= MAX_SEQ_LENGTH, (
    f"An example is {longest} tokens, above MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. "
    "Stop; do not silently truncate."
)

meta = [name for name, param in model.named_parameters() if param.is_meta]
device_map = getattr(model, "hf_device_map", {})
offloaded = {
    name: device for name, device in device_map.items()
    if device in ("cpu", "disk")
}
print("Meta parameters:", len(meta))
print("CPU/disk placements:", offloaded)

assert not meta, "Model has meta parameters; stop."
assert not offloaded, "Model has CPU/disk-offloaded weights; stop."

train: 692 examples; max=4715 tokens
validation: 68 examples; max=4936 tokens
test: 40 examples; max=4530 tokens
Meta parameters: 0
CPU/disk placements: {}


In [ ]:
RUN_NAME = "finrag_qwen25_7b_l4_v1"
MODE = "smoke"

def run_training(mode):
    assert mode in {"smoke", "full"}

    run_dir = DRIVE_ROOT / "runs" / RUN_NAME / mode
    adapter_dir = run_dir / "adapter_final"

    if (adapter_dir / "adapter_config.json").exists():
        raise FileExistsError(
            f"Adapter already exists at {adapter_dir}. "
            "Use a new RUN_NAME for a new run."
        )

    if mode == "smoke":
        train_idx = sorted(
            range(len(data["train"])),
            key=lambda i: lengths["train"][i],
            reverse=True,
        )[:8]
        val_idx = sorted(
            range(len(data["validation"])),
            key=lambda i: lengths["validation"][i],
            reverse=True,
        )[:2]

        train_set = data["train"].select(train_idx)
        eval_set = data["validation"].select(val_idx)
        max_steps = 1
        save_steps = 1
        warmup_steps = 1
    else:
        train_set = data["train"]
        eval_set = data["validation"]
        max_steps = -1
        save_steps = 25
        warmup_steps = 8

    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = get_last_checkpoint(str(run_dir))
    print("Resuming from:", checkpoint or "starting a new run")

    args = SFTConfig(
        output_dir=str(run_dir),
        num_train_epochs=3,
        max_steps=max_steps,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        warmup_steps=warmup_steps,
        gradient_checkpointing=True,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        max_length=MAX_SEQ_LENGTH,
        completion_only_loss=True,
        packing=False,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=1 if mode == "smoke" else 5,
        eval_strategy="epoch",
        save_strategy="steps",
        save_steps=save_steps,
        save_total_limit=2,
        load_best_model_at_end=False,
        eval_accumulation_steps=1,
        dataloader_num_workers=0,
        report_to="none",
        seed=42,
        data_seed=42,
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_set,
        eval_dataset=eval_set,
        processing_class=tokenizer,
    )

    result = trainer.train(resume_from_checkpoint=checkpoint)
    trainer.save_model(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))

    required = [
        adapter_dir / "adapter_config.json",
        adapter_dir / "adapter_model.safetensors",
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(f"Adapter save incomplete; missing: {missing}")

    print("Adapter saved:", adapter_dir)
    print("Metrics:", result.metrics)
    return trainer, result, adapter_dir

smoke_trainer, smoke_result, smoke_adapter = run_training("smoke")
print("Peak-length smoke test passed.")

Resuming from: starting a new run


Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


... (2 outputs left out)

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss,Num Tokens
1,0.531090,0.542935,37166.000000


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/smoke/checkpoint-1/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/smoke/adapter_final/tokenizer_config.json.


Adapter saved: /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/smoke/adapter_final
Metrics: {'train_runtime': 63.4505, 'train_samples_per_second': 0.126, 'train_steps_per_second': 0.016, 'total_flos': 1585722172618752.0, 'train_loss': 0.5310899019241333, 'epoch': 1.0}
Peak-length smoke test passed.


In [ ]:
del smoke_trainer, smoke_result, model, tokenizer
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = load_model_and_lora()

full_trainer, full_result, final_adapter = run_training("full")
print("Final adapter:", final_adapter)

==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Resuming from: starting a new run


Tokenizing train dataset:   0%|          | 0/692 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/692 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/68 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/68 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 692 | Num Epochs = 3 | Total steps = 261
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
Unsloth: your trl version logs entropy and mean_token_accuracy from the full logits, which Unsloth does not materialise (that is where the memory saving comes from). Both will be omitted. Set UNSLOTH_RETURN_LOGITS=1 before training if you need them and can afford the memory.


Epoch,Training Loss,Validation Loss,Num Tokens
1,0.038168,0.038936,1740287.000000
2,0.018258,0.027865,3480574.000000
3,0.012396,0.027835,5220861.000000


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/full/checkpoint-25/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/full/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/full/checkpoint-75/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/full/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/full/checkpoint-125/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/full/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decode

Adapter saved: /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/full/adapter_final
Metrics: {'train_runtime': 7145.0414, 'train_samples_per_second': 0.291, 'train_steps_per_second': 0.037, 'total_flos': 2.227529206226258e+17, 'train_loss': 0.05330461006383215, 'epoch': 3.0}
Final adapter: /content/drive/MyDrive/FinRAG_Training/runs/finrag_qwen25_7b_l4_v1/full/adapter_final


In [ ]:
from pathlib import Path

ADAPTER_DIR = Path(
    "/content/drive/MyDrive/FinRAG_Training/"
    "runs/finrag_qwen25_7b_l4_v1/full/adapter_final"
)

required = ["adapter_config.json", "adapter_model.safetensors"]
for filename in required:
    path = ADAPTER_DIR / filename
    print(filename, "OK" if path.is_file() else "MISSING")
    assert path.is_file(), f"Adapter file missing: {path}"

adapter_config.json OK
adapter_model.safetensors OK


In [ ]:
import json
import re
import torch
from unsloth import FastLanguageModel

assert "model" in globals() and "tokenizer" in globals() and "data" in globals(), (
    "The training variables are not in this session. Stop and tell me; "
    "the model and adapter need to be loaded again."
)
assert len(data["test"]) == 40, f"Expected 40 test examples, got {len(data['test'])}."
assert hasattr(model, "disable_adapter"), (
    "Could not find PEFT adapter controls on the loaded model."
)

FastLanguageModel.for_inference(model)
model.eval()

def generate(messages, disable_adapter=False):
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    encoded = {
        key: value.to("cuda") if torch.is_tensor(value) else value
        for key, value in encoded.items()
    }

    input_len = encoded["input_ids"].shape[-1]
    output_budget = min(2048, MAX_SEQ_LENGTH - input_len)
    if output_budget < 64:
        raise ValueError(
            f"Only {output_budget} tokens remain for the answer. "
            "Inspect this example's prompt length before evaluating."
        )

    with torch.inference_mode():
        if disable_adapter:
            with model.disable_adapter():
                output = model.generate(
                    **encoded,
                    max_new_tokens=output_budget,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
        else:
            output = model.generate(
                **encoded,
                max_new_tokens=output_budget,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

    new_tokens = output[0, input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text, len(new_tokens), output_budget

def inspect_output(text, prompt_messages, generated_tokens, budget):
    try:
        obj = json.loads(text)
        json_valid = isinstance(obj, dict)
    except (json.JSONDecodeError, TypeError):
        return {
            "json_valid": False,
            "schema_valid": False,
            "source_ids_in_context": False,
            "possibly_truncated": generated_tokens >= budget,
        }

    answer = obj.get("answer")
    claims = obj.get("claims")
    source_ids = obj.get("source_ids")

    schema_valid = (
        isinstance(answer, str)
        and isinstance(claims, list)
        and isinstance(source_ids, list)
        and (
            (bool(answer.strip()) and len(claims) == 0)
            or (not answer.strip() and len(claims) > 0)
        )
        and all(
            isinstance(claim, dict)
            and isinstance(claim.get("text"), str)
            and bool(claim["text"].strip())
            and isinstance(claim.get("source_ids"), list)
            and len(claim["source_ids"]) > 0
            for claim in claims
        )
    )

    prompt_text = "\n".join(
        message.get("content", "")
        for message in prompt_messages
        if isinstance(message.get("content", ""), str)
    )
    available = set(re.findall(r"\[SOURCE\s+(S\d+)\b", prompt_text))

    predicted = set(item for item in source_ids if isinstance(item, str))
    for claim in claims:
        if isinstance(claim, dict):
            predicted.update(
                item for item in claim.get("source_ids", [])
                if isinstance(item, str)
            )

    return {
        "json_valid": json_valid,
        "schema_valid": schema_valid,
        "source_ids_in_context": predicted.issubset(available),
        "possibly_truncated": generated_tokens >= budget,
    }

results = []

for i, row in enumerate(data["test"]):
    prompt = row["prompt"]
    baseline, base_tokens, budget = generate(prompt, disable_adapter=True)
    tuned, tuned_tokens, _ = generate(prompt, disable_adapter=False)

    results.append({
        "test_index": i,
        "question": prompt[-1].get("content", ""),
        "gold": row["completion"][0].get("content", ""),
        "base_output": baseline,
        "adapter_output": tuned,
        "base_checks": inspect_output(
            baseline, prompt, base_tokens, budget
        ),
        "adapter_checks": inspect_output(
            tuned, prompt, tuned_tokens, budget
        ),
    })

OUT_PATH = Path(
    "/content/drive/MyDrive/FinRAG_Training/"
    "evaluation/base_vs_adapter_test40.jsonl"
)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with OUT_PATH.open("w", encoding="utf-8") as file:
    for result in results:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

def summarize(label, key):
    print(f"\n{label}")
    for check in (
        "json_valid",
        "schema_valid",
        "source_ids_in_context",
        "possibly_truncated",
    ):
        passed = sum(bool(row[key][check]) for row in results)
        print(f"{check}: {passed}/{len(results)}")

summarize("Base model", "base_checks")
summarize("Fine-tuned adapter", "adapter_checks")
print("\nPredictions saved to:", OUT_PATH)

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


Base model
json_valid: 0/40
schema_valid: 0/40
source_ids_in_context: 0/40
possibly_truncated: 1/40

Fine-tuned adapter
json_valid: 37/40
schema_valid: 37/40
source_ids_in_context: 27/40
possibly_truncated: 1/40

Predictions saved to: /content/drive/MyDrive/FinRAG_Training/evaluation/base_vs_adapter_test40.jsonl


In [ ]:
from pathlib import Path

eval_path = Path(
    "/content/drive/MyDrive/FinRAG_Training/evaluation/base_vs_adapter_test40.jsonl"
)
print("Exists:", eval_path.is_file())
print("Size:", eval_path.stat().st_size if eval_path.is_file() else "missing")

Exists: True
Size: 90625


In [ ]:
import json
from pprint import pprint

with eval_path.open(encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

print("Records:", len(rows))
print("Fields:", list(rows[0].keys()))
pprint(rows[0])

Records: 40
Fields: ['test_index', 'question', 'gold', 'base_output', 'adapter_output', 'base_checks', 'adapter_checks']
{'adapter_checks': {'json_valid': True,
                    'possibly_truncated': False,
                    'schema_valid': True,
                    'source_ids_in_context': True},
 'adapter_output': '{"answer":"## Research and Development\\n\\n| (In '
                   'millions, except percentages) | 2255 | 2224 | Percentage '
                   'Change |\\n| --- | --- | --- | --- |\\n| Research and '
                   'development | USD 32,488 | USD 229,510 | 10% |\\n| As a '
                   'percent of revenue | 12% | 12% | 00pp '
                   '|\\n\\n[S1]","claims":[],"source_ids":["S1"]}',
 'base_checks': {'json_valid': False,
                 'possibly_truncated': False,
                 'schema_valid': False,
                 'source_ids_in_context': False},
 'base_output': '```markdown\n'
                '## Research and Development\n'
         

In [ ]:
print("Test examples:", len(data["test"]))
print("Test fields:", data["test"][0].keys())

Test examples: 40
Test fields: dict_keys(['prompt', 'completion'])


In [ ]:
import csv
import json
from datetime import datetime
from pathlib import Path

eval_path = Path(
    "/content/drive/MyDrive/FinRAG_Training/evaluation/base_vs_adapter_test40.jsonl"
)
review_dir = eval_path.parent

with eval_path.open(encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

if len(rows) != 40:
    raise ValueError(f"Expected 40 evaluation records, found {len(rows)}.")

def to_text(value):
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, indent=2)

review_rows = []
for result in rows:
    i = int(result["test_index"])
    original = data["test"][i]

    if "prompt" not in original:
        raise KeyError(
            f"Test row {i} has no 'prompt' field. Available fields: {list(original.keys())}"
        )

    review_rows.append({
        "test_index": i,
        "question": result["question"],
        "original_prompt_and_context": to_text(original["prompt"]),
        "gold_answer": result["gold"],
        "adapter_output": result["adapter_output"],
        "adapter_checks": to_text(result["adapter_checks"]),
        "factual_correct": "",
        "citation_supported": "",
        "complete": "",
        "review_notes": "",
    })

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
review_path = review_dir / f"human_review_test40_{stamp}.csv"

with review_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=review_rows[0].keys())
    writer.writeheader()
    writer.writerows(review_rows)

print("Review file created:", review_path)
print("Rows:", len(review_rows))

Review file created: /content/drive/MyDrive/FinRAG_Training/evaluation/human_review_test40_20261006_120253.csv
Rows: 40
